### Student Perfomance

Notebook para experimentar con este problema cargando datos desde s3 y registrando en mlflow

In [8]:
# imports
import optuna
import pandas as pd
import mlflow
import sklearn
import awswrangler as wr
from dotenv import load_dotenv
import os
from pathlib import Path
from helpers.s3_helpers import get_latest_s3_folder
from helpers.model_helpers import logistic_regression_model_train_log, svc_model_training
import sweetviz as sv
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from mlflow import MlflowClient
from mlflow.exceptions import RestException

In [ ]:
# load env variables
load_dotenv()

In [ ]:
# connect to mlflow 
mlflow.set_tracking_uri("http://localhost:5001")

In [ ]:
# Use most recent data
latest_path = get_latest_s3_folder(
    'data',
    'student_performance',
    '%Y-%m-%d_%H-%M-%S'
)
print(latest_path)

In [ ]:
# Load data from bucket
X_train =  wr.s3.read_csv(latest_path+'X_train.csv')
y_train =  wr.s3.read_csv(latest_path+'y_train.csv')

X_test =  wr.s3.read_csv(latest_path+'X_test.csv')
y_test =  wr.s3.read_csv(latest_path+'y_test.csv')

In [ ]:
# Train and log a logistic regression
best_model, study, best_params = logistic_regression_model_train_log(
    X_train,
    y_train,
    X_test,
    y_test,
    experiment_name='Student Performance',
    model_name = 'student_performance_logreg',
    n_trials=100,
    cv=5,
    scoring='f1',
    random_state=42
)

In [10]:
# Register the model

client = MlflowClient()
name = "student_performance_logreg_prod"
desc = "Classifier for final student grade"

# Creamos el modelo productivo si no existe previamente
try:
    client.create_registered_model(name=name, description=desc)
except RestException:
    pass

# Guardamos como tag los hiper-parámetros en la versión del modelo
tags = best_model.get_params()
tags["model"] = type(best_model).__name__

# Guardamos la versión del modelo
result = client.create_model_version(
    name=name,
    source='models',
    tags=tags
)

# Y creamos la versión con el alias de champion
client.set_registered_model_alias(name, "champion", result.version)

RestException: INVALID_PARAMETER_VALUE: Invalid model version source: 'models'. To use a local path as a model version source, the run_id request parameter has to be specified and the local path has to be contained within the artifact directory of the run specified by the run_id.